The pipeline begins by importing essential libraries including pandas, seaborn, matplotlib, scikit-learn, xgboost, and shap. It loads the Titanic dataset and applies a powerful custom feature engineering function. This function extracts the passenger's title from the name, creates new features such as family size, cabin presence, age/fare bins, interaction terms (like age multiplied by class), and target encodes the title based on survival rates. Missing values are handled smartly using grouped medians and modes.

Once the dataset is prepared, it performs exploratory visualizations to understand survival distributions across sex, age, and fare, followed by a heatmap to inspect feature correlations. The core features are selected, and three different models are defined: an XGBoost classifier with tuned hyperparameters, a Random Forest classifier, and a Logistic Regression model with increased iteration limits to ensure convergence.

These models are combined using a soft-voting ensemble classifier, and the pipeline evaluates the ensemble using 5-fold cross-validation, reporting both the mean and standard deviation of accuracy.

Afterward, the test dataset undergoes the same feature engineering and one-hot encoding steps, ensuring all necessary columns are aligned with the training set. The ensemble model is trained on the full dataset, and predictions are generated on the test set for submission to Kaggle.

Finally, the model interpretability is enhanced using SHAP (SHapley Additive exPlanations), where a TreeExplainer visualizes feature importance, helping to understand which features influenced survival predictions the most.

<font color = "DeepSkyBlue">**Import libraries**

This section initializes the Titanic machine learning pipeline by importing essential Python libraries. It includes tools for data handling (pandas, numpy), visualization (seaborn, matplotlib), model building (RandomForestClassifier, LogisticRegression, XGBClassifier, VotingClassifier), preprocessing (StandardScaler, SimpleImputer, OneHotEncoder, ColumnTransformer, Pipeline), and model explainability (shap). These libraries form the backbone of the end-to-end pipeline, supporting everything from feature engineering and training to ensemble modeling and interpretability.

In [ ]:
# === Titanic ML Pipeline: Boosted Feature Engineering & XGBoost + Ensemble ===
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestClassifier, VotingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from xgboost import XGBClassifier
import shap

<font color = "DeepSkyBlue">**Load data**

This section loads the Titanic dataset from Kaggle into two separate DataFrames: train_df for training the model and test_df for generating predictions. It uses pandas.read_csv() to read the CSV files into memory for further processing and modeling.

In [ ]:
# === Load data ===
train_df = pd.read_csv("/kaggle/input/titanic/train.csv")
test_df = pd.read_csv("/kaggle/input/titanic/test.csv")

<font color = "DeepSkyBlue">**Feature Engineering Function**

This function performs comprehensive feature engineering on the Titanic dataset. It extracts the passenger's title from the name and standardizes rare or alternative titles. It then creates a target-encoded version of the title (Title_TE) based on survival rates. Additional features are engineered, such as FamilySize, IsAlone, and HasCabin, as well as interaction features like Age*Class and Sex*Pclass. Missing values in categorical and numerical columns are imputed, and new binned variables (AgeBin, FareBin) are created to help models capture non-linear relationships.

In [ ]:
# === Feature Engineering Function ===
def engineer_features(df):
    df = df.copy()
    df['Title'] = df['Name'].str.extract(' ([A-Za-z]+)\\.', expand=False)
    df['Title'] = df['Title'].replace(
        ['Lady', 'Countess', 'Capt', 'Col', 'Don', 'Dr', 'Major', 
         'Rev', 'Sir', 'Jonkheer', 'Dona'], 'Rare')
    df['Title'] = df['Title'].replace({'Mlle': 'Miss', 'Ms': 'Miss', 'Mme': 'Mrs'})
    if 'Survived' in df.columns:
        title_map = df.groupby('Title')['Survived'].mean().to_dict()
    else:
        title_map = {'Mr': 0.16, 'Miss': 0.70, 'Mrs': 0.79, 'Master': 0.57, 'Rare': 0.47}
    df['Title_TE'] = df['Title'].map(title_map)
    df['FamilySize'] = df['SibSp'] + df['Parch'] + 1
    df['IsAlone'] = (df['FamilySize'] == 1).astype(int)
    df['HasCabin'] = df['Cabin'].notnull().astype(int)
    df['Embarked'] = df['Embarked'].fillna(df['Embarked'].mode()[0])
    df['Fare'] = df['Fare'].fillna(df['Fare'].median())
    df['Sex'] = df['Sex'].map({'male': 0, 'female': 1})
    df['Embarked'] = df['Embarked'].map({'S': 0, 'C': 1, 'Q': 2})
    df['Age'] = df['Age'].fillna(df.groupby('Title')['Age'].transform('median'))
    df['AgeBin'] = pd.cut(df['Age'], bins=[0, 12, 18, 35, 50, 80], labels=False)
    df['FareBin'] = pd.qcut(df['Fare'], 4, labels=False)
    df['Age*Class'] = df['Age'] * df['Pclass']
    df['Fare_per_person'] = df['Fare'] / df['FamilySize']
    df['Sex*Pclass'] = df['Sex'] * df['Pclass']
    return df

<font color = "DeepSkyBlue">**Process Train Data**

This section applies the previously defined feature engineering function to the training dataset and then performs one-hot encoding on the Title column to convert categorical title information into binary features. The drop_first=True option avoids multicollinearity by dropping the first category in each encoded group.

In [ ]:
# === Process Train Data ===
df = engineer_features(train_df)
df = pd.get_dummies(df, columns=['Title'], drop_first=True)

<font color = "DeepSkyBlue">**Visualizations**

This section generates exploratory visualizations to understand how features relate to survival. It includes a count plot of survival by sex, a histogram of age distributions colored by survival status, and a boxplot comparing fare across survival outcomes. Finally, a correlation heatmap is created for all selected features, helping identify strong predictors of survival and potential multicollinearity.

In [ ]:
# === Visualizations ===
sns.set(style="whitegrid")
plt.figure(figsize=(8, 4))
sns.countplot(x="Survived", hue="Sex", data=df)
plt.title("Survival Count by Sex")
plt.show()

plt.figure(figsize=(8, 4))
sns.histplot(data=df, x="Age", bins=30, hue="Survived", kde=False, multiple="stack")
plt.title("Age Distribution by Survival")
plt.show()

plt.figure(figsize=(8, 4))
sns.boxplot(x="Survived", y="Fare", data=df)
plt.title("Fare vs Survival")
plt.show()

features = ['Pclass', 'Sex', 'Age', 'Fare', 'Embarked', 'FamilySize', 'IsAlone', 'HasCabin', 'AgeBin', 'FareBin', 'Age*Class', 'Fare_per_person', 'Sex*Pclass', 'Title_TE'] + [col for col in df.columns if col.startswith('Title_') and col != 'Title_TE']
plt.figure(figsize=(10, 6))
sns.heatmap(df[features + ['Survived']].corr(), annot=True, cmap="coolwarm")
plt.title("Feature Correlation Heatmap")
plt.show()

<font color = "DeepSkyBlue">**Train-Validation Split**

This part separates the processed dataset into features (X) and the target variable (y, which is Survived) to prepare for model training and validation.

In [ ]:
# === Train-Validation Split ===
X = df[features]
y = df['Survived']

<font color = "DeepSkyBlue">**Define Models**

This section defines three machine learning models to be used in the ensemble: an XGBoost classifier with fine-tuned hyperparameters for boosting performance, a Random Forest classifier for robust ensemble-based predictions, and a Logistic Regression model with increased iteration limits to ensure convergence.

In [ ]:
# === Define Models ===
xgb_model = XGBClassifier(n_estimators=600, max_depth=3, learning_rate=0.015, 
                          subsample=0.85, colsample_bytree=0.85,
                          reg_alpha=0.1, reg_lambda=1.0,
                          use_label_encoder=False, eval_metric='logloss', random_state=42)

rf_model = RandomForestClassifier(n_estimators=500, max_depth=6, random_state=42)
lr_model = LogisticRegression(max_iter=3000)

<font color = "DeepSkyBlue">**Voting Classifier Ensemble**

This section builds a soft voting ensemble classifier that combines predictions from XGBoost, Random Forest, and Logistic Regression models. It evaluates the ensemble’s performance using 5-fold cross-validation and reports the mean and standard deviation of the accuracy, providing a robust estimate of generalization performance.

In [ ]:
# === Voting Classifier Ensemble ===
ensemble = VotingClassifier(estimators=[
    ('xgb', xgb_model),
    ('rf', rf_model),
    ('lr', lr_model)
], voting='soft')

cv_scores = cross_val_score(ensemble, X, y, cv=5, scoring='accuracy')
print(f"Ensemble CV Accuracy: {cv_scores.mean():.5f} ± {cv_scores.std():.5f}")

<font color = "DeepSkyBlue">**Process Test Set**

This section prepares the test dataset by applying the same feature engineering steps used on the training data. It also ensures that all dummy variables related to 'Title' match the training set by adding any missing columns with default values, maintaining consistency in the feature space for model prediction.

In [ ]:
# === Process Test Set ===
test = engineer_features(test_df)
test = pd.get_dummies(test, columns=['Title'], drop_first=True)
for col in [c for c in df.columns if c.startswith('Title_')]:
    if col not in test.columns:
        test[col] = 0

<font color = "DeepSkyBlue">**Align columns**

This line ensures that the test dataset (X_test) has the exact same columns (features) in the same order as the training dataset (X). This alignment is essential before making predictions using the trained model.

In [ ]:
# Align columns
X_test = test[features]

<font color = "DeepSkyBlue">**Final Training & Prediction**

In this step, the XGBoost model is first trained separately so that SHAP can later explain its predictions. Then, the full voting ensemble model (combining XGBoost, Random Forest, and Logistic Regression) is trained on the entire dataset. Finally, predictions are generated for the test set using the trained ensemble.

In [ ]:
# === Final Training & Prediction ===
xgb_model.fit(X, y)  # For SHAP
final_model = ensemble.fit(X, y)
test_preds = final_model.predict(X_test)

<font color = "DeepSkyBlue">**Submission**

This block prepares the final submission file by pairing each test passenger's ID with the predicted survival outcome. The results are saved as a CSV file named titanic_boosted_submission.csv and a confirmation message is printed upon completion.

In [ ]:
# === Submission ===
submission = pd.DataFrame({
    'PassengerId': test['PassengerId'],
    'Survived': test_preds
})
submission.to_csv("/kaggle/working/titanic_boosted_submission.csv", index=False)
print("Submission saved: titanic_boosted_submission.csv")


<font color = "DeepSkyBlue">**SHAP Feature Importance Visualization**

This section uses SHAP (SHapley Additive exPlanations) to interpret the XGBoost model. A TreeExplainer is created to calculate SHAP values for all features, and then a summary bar plot visualizes the most impactful features influencing survival predictions.

In [ ]:
# === SHAP Feature Importance Visualization ===
explainer = shap.TreeExplainer(xgb_model)
shap_values = explainer.shap_values(X)
shap.summary_plot(shap_values, X, plot_type="bar")